<a href="https://colab.research.google.com/github/Law2030/apps/blob/main/ollama_colab_free_server_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An LLM server that runs Ollama on Google Colab's GPU and exposes the Ollama endpoint via an ngrok tunnel.

Key Features
- ✅ Completely free. No data sent to external APIs — privacy protected via local inference
- 🎛️ Select a model via UI; pull and launch are fully automated
- 🚀 Instantly connectable from Continue or Claude Code


> ※ Append `/v1` to the base URL for Codex and other OpenAI-compatible clients. Example: `https://xxxx.ngrok-free.app/v1`

In [4]:
#@title 📋 Model Registry

# @markdown ### Model Configuration
# @markdown > Enter the model names you want to add, separated by commas. Guide: `8B` recommended · `14B` feasible · `20B+` slow
# @markdown > - Find official model names here: https://ollama.com/search

model_list = "qwen3:8b, qwen3:14b, qwen2.5-coder:7b, qwen2.5-coder:14b, ministral-3:8b, ministral-3:14b, devstral-small-2:24b, gpt-oss:20b, deepseek-r1:8b, glm-4.7-flash, qwen3.5:9b, qwen3.5:4b, qwen3.5:2b, qwen3.5:0.8b, nemotron-3-nano:4b, nemotron-3-nano:30b" #@param {type:"string"}

# @markdown ### context length (num_ctx)
# @markdown > T4 (16GB VRAM) Guide
# @markdown > | Model Size | Rec. ctx | Notes |
# @markdown > |---|---|---|
# @markdown > | ~4B  | 32768 | Plentiful |
# @markdown > | ~8B  | 16384 | Standard |
# @markdown > | ~14B | 8192  | VRAM tight |
# @markdown > | ~20B+ | 4096 | Min. usage |
# @markdown >
# @markdown > Default (0) lets Ollama decide (equivalent to 4096).
num_ctx = 16384 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ Model list is empty. Please enter at least one model.")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 Select a Model</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    'Select one model to launch, then run the next cell.</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(padding='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))

print(f"✅ Model list loaded: {len(AVAILABLE_MODELS)} model(s)")
print("➡️ After selecting a model, run the next cell (Server).")


✅ Model list loaded: 16 model(s)
➡️ After selecting a model, run the next cell (Server).


In [6]:

# ============================================================
# 🚀 OLLAMA COLAB SERVER — OPTIMIZED FOR TESLA T4
# Qwen 3.5 9B | 12K Context | ngrok
# ============================================================

import os
import sys
import time
import shutil
import subprocess
import requests
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

OLLAMA_HOST = "0.0.0.0:11434"
OLLAMA_LOCAL_URL = "http://127.0.0.1:11434"

# Select model from your Model Registry cell.
try:
    SELECTED_MODEL = str(model_selector.value).strip()
except NameError:
    raise RuntimeError(
        "Run the Model Registry cell first and select a model."
    )

if not SELECTED_MODEL:
    raise RuntimeError("No model selected.")

# Use at least 12K context.
# If your Model Registry already selects a higher value, preserve it.
try:
    REGISTRY_CONTEXT = int(num_ctx)
except (NameError, TypeError, ValueError):
    REGISTRY_CONTEXT = 12288

SELECTED_CONTEXT = max(REGISTRY_CONTEXT, 12288)

# T4-friendly settings
KEEP_ALIVE = "24h"
KV_CACHE_TYPE = "q8_0"
NUM_PARALLEL = "1"

print("=" * 60)
print("🚀 OLLAMA COLAB SERVER")
print("=" * 60)

print("Model:", SELECTED_MODEL)
print("Context:", SELECTED_CONTEXT)
print("KV cache:", KV_CACHE_TYPE)
print("Parallel requests:", NUM_PARALLEL)


# ============================================================
# 2. SYSTEM DEPENDENCIES
# ============================================================

print("\n◦ Installing system dependencies...")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd", "curl"],
    check=True
)

print("✓ System dependencies ready")


# ============================================================
# 3. INSTALL OLLAMA
# ============================================================

print("\n◦ Installing Ollama...")

if shutil.which("ollama") is None:
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True,
        check=True
    )

print("✓ Ollama installed")


# ============================================================
# 4. INSTALL PYNGROK
# ============================================================

print("\n◦ Installing pyngrok...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "pyngrok"
    ],
    check=True
)

from pyngrok import ngrok

print("✓ pyngrok ready")


# ============================================================
# 5. GPU CHECK
# ============================================================

print("\n◦ Checking GPU...")

try:
    import torch

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = (
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3)
        )

        print("✓ GPU:", gpu_name)
        print(f"✓ VRAM: {gpu_memory:.2f} GB")

    else:
        print("⚠️ CUDA GPU not detected.")

except Exception as error:
    print("⚠️ GPU check unavailable:", error)

try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception:
    pass


# ============================================================
# 6. CONFIGURE OLLAMA ENVIRONMENT
# ============================================================

print("\n◦ Configuring Ollama...")

server_env = os.environ.copy()

server_env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_KEEP_ALIVE": KEEP_ALIVE,
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_NUM_PARALLEL": NUM_PARALLEL,
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": KV_CACHE_TYPE,
    "OLLAMA_MAX_QUEUE": "16",
    "OLLAMA_LOAD_TIMEOUT": "10m"
})

# ============================================================
# 7. START OLLAMA SERVER
# ============================================================

print("\n◦ Starting Ollama server...")

# Stop an old Ollama server so the new environment is applied.
subprocess.run(
    ["pkill", "-f", "^ollama serve$"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=server_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

server_ready = False

for attempt in range(60):
    try:
        response = requests.get(
            f"{OLLAMA_LOCAL_URL}/api/tags",
            timeout=5
        )

        if response.status_code == 200:
            server_ready = True
            break

    except requests.RequestException:
        pass

    if ollama_process.poll() is not None:
        raise RuntimeError(
            "Ollama server exited unexpectedly."
        )

    time.sleep(2)

if not server_ready:
    raise RuntimeError(
        "Ollama server did not become ready."
    )

print("✓ Ollama server ready")


# ============================================================
# 8. DOWNLOAD / VERIFY MODEL
# ============================================================

print(f"\n◦ Checking model: {SELECTED_MODEL}")

response = requests.get(
    f"{OLLAMA_LOCAL_URL}/api/tags",
    timeout=20
)

response.raise_for_status()

existing_models = response.json().get("models", [])

model_exists = any(
    item.get("name") == SELECTED_MODEL
    or item.get("model") == SELECTED_MODEL
    for item in existing_models
)

if not model_exists:

    print("◦ Downloading model...")

    pull_process = subprocess.Popen(
        ["ollama", "pull", SELECTED_MODEL],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in pull_process.stdout:
        line = line.strip()
        if line:
            print(line)

    pull_process.wait()

    if pull_process.returncode != 0:
        raise RuntimeError(
            f"Model download failed: {SELECTED_MODEL}"
        )

print("✓ Model downloaded / available")


# ============================================================
# 9. VERIFY MODEL
# ============================================================

print("\n◦ Verifying model...")

show_response = requests.post(
    f"{OLLAMA_LOCAL_URL}/api/show",
    json={"name": SELECTED_MODEL},
    timeout=60
)

if show_response.status_code != 200:
    raise RuntimeError(
        "Model verification failed: "
        + show_response.text
    )

print("✓ Model verified")


# ============================================================
# 10. WARM UP MODEL WITH 12K CONTEXT
# ============================================================

print("\n◦ Loading model into GPU / memory...")

warmup_started = time.time()

warmup_response = requests.post(
    f"{OLLAMA_LOCAL_URL}/api/generate",
    json={
        "model": SELECTED_MODEL,
        "prompt": "Reply with the word READY.",
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "options": {
            "num_ctx": SELECTED_CONTEXT,
            "num_predict": 32,
            "temperature": 0
        }
    },
    timeout=600
)

warmup_response.raise_for_status()

warmup_data = warmup_response.json()

print("✓ Model loaded")
print("✓ Warm-up time:", round(time.time() - warmup_started, 2), "seconds")
print("✓ Warm-up response:", warmup_data.get("response", "").strip())


# ============================================================
# 11. CHECK GPU ALLOCATION
# ============================================================

print("\n◦ Checking Ollama model allocation...")

try:
    ps_response = requests.get(
        f"{OLLAMA_LOCAL_URL}/api/ps",
        timeout=20
    )

    ps_response.raise_for_status()

    running_models = ps_response.json().get("models", [])

    for model in running_models:
        print("Model:", model.get("name"))
        print(
            "VRAM allocation:",
            round(model.get("size_vram", 0) / (1024 ** 3), 2),
            "GB"
        )
        print(
            "Total model size:",
            round(model.get("size", 0) / (1024 ** 3), 2),
            "GB"
        )

    if not running_models:
        print("⚠️ No loaded model reported by /api/ps.")

except Exception as error:
    print("⚠️ Allocation check failed:", error)


# ============================================================
# 12. PREPARE NGROK
# ============================================================

print("\n◦ Preparing ngrok...")

try:
    ngrok.kill()
    time.sleep(2)
except Exception:
    pass

# Reuse an already configured token.
# If no token exists, ask securely without displaying it.

ngrok_config = Path.home() / ".config" / "ngrok" / "ngrok.yml"

if not ngrok_config.exists():
    from getpass import getpass

    token = getpass("Enter your NEW ngrok authtoken: ").strip()

    if not token:
        raise RuntimeError("ngrok authtoken is required.")

    ngrok.set_auth_token(token)

    del token

print("✓ ngrok configuration ready")


# ============================================================
# 13. CREATE PUBLIC TUNNEL
# ============================================================

print("\n◦ Creating ngrok tunnel...")

try:
    tunnel = ngrok.connect(
        addr=11434,
        proto="http"
    )

    OLLAMA_PUBLIC_URL = tunnel.public_url.rstrip("/")

except Exception as error:
    raise RuntimeError(
        "ngrok tunnel creation failed. "
        "Check your ngrok account and authtoken. "
        f"Details: {error}"
    )

print("✓ Public tunnel established")


# ============================================================
# 14. TEST PUBLIC ENDPOINT
# ============================================================

print("\n◦ Testing public endpoint...")

try:
    public_test = requests.get(
        f"{OLLAMA_PUBLIC_URL}/api/tags",
        timeout=30
    )

    public_test.raise_for_status()

    print("✓ Public API accessible")

except Exception as error:
    raise RuntimeError(
        f"Public endpoint test failed: {error}"
    )


# ============================================================
# 15. FINAL CONFIGURATION
# ============================================================

print("\n" + "=" * 60)
print("🎉 OLLAMA SERVER READY")
print("=" * 60)

print("\nMODEL CONFIGURATION")
print("-" * 60)

print("Model:", SELECTED_MODEL)
print("Context:", SELECTED_CONTEXT)
print("KV cache:", KV_CACHE_TYPE)
print("Parallel:", NUM_PARALLEL)
print("Keep alive:", KEEP_ALIVE)

print("\nENDPOINTS")
print("-" * 60)

print("Local:", OLLAMA_LOCAL_URL)
print("Public:", OLLAMA_PUBLIC_URL)

print("\nVERCEL ENVIRONMENT VARIABLES")
print("-" * 60)

print(f"OLLAMA_URL={OLLAMA_PUBLIC_URL}")
print(f"OLLAMA_MODEL={SELECTED_MODEL}")
print(f"OLLAMA_NUM_CTX={SELECTED_CONTEXT}")

print("\nTEST")
print("-" * 60)

print(f"{OLLAMA_PUBLIC_URL}/api/tags")

print("\nIMPORTANT")
print("-" * 60)

print("1. Keep this Colab runtime running.")
print("2. Do not expose confidential contracts through")
print("   an unauthenticated public Ollama endpoint.")
print("3. Your Next.js API must also request num_ctx=",
      SELECTED_CONTEXT)
print("4. Keep num_predict at 3072 initially.")
print("5. Check /api/ps to verify GPU allocation.")

print("=" * 60)



OLLAMA COLAB SERVER 🚀
──────────────────────────────────────────
  ◦ System   Installing dependencies (zstd)
  ◦ System   Installing Ollama
  ◦ System   Installing pyngrok
  ✓ Ready    Ollama server started (0.0.0.0:11434)
  ◦ Network  Establishing ngrok tunnel (port: 11434)


PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'